<div style="background-color: black;">
<hr style="border: 3px solid skyblue;">
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 20px; font-family: TimesNewRoman; color: skyblue">
    TIME SERIES DATA PROCESSING
<br>
    CROSS BORDER FLOWS
</div>
<div style="text-align: center; margin-left: 0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color: skyblue">
    Main Formatting Notebook
    <br>
    from PYPSA to DISPA-SET
</div>
<div style="text-align: justify; margin-left: 0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color: skyblue">
This script processes raw PyPSA simulation data to generate cross-border flow time series for every country modeled in the Dispa-SET Unleashed project.
<br>
Refer to the explanation cells for a step-by-step guide from raw data processing to final output.
</div>
    <hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    1. Notebook Set Up
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Importing needed libraries.
<hr style="border: 1px solid skyblue;">
</div>
</div>

In [1]:
import os
import csv
from datetime import datetime
import requests
import pandas as pd
from shutil import move
import numpy as np
import shutil
from bs4 import BeautifulSoup
import re
import io
import plotly.graph_objects as go
from typing import List, Dict, Tuple, Optional, Tuple
import re
from IPython.display import HTML
from difflib import get_close_matches
from collections import defaultdict
import warnings
from pathlib import Path
import glob
from itertools import permutations
from itertools import combinations

<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Auxiliar Code
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cell has the purpose to create the correponding folders with the name of all the EU countries available in the ENTSOE data base.
    <br>
    Uncomment it to use it just if needed.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [2]:
'''
# List of countries with their acronyms in parentheses
countries = [
    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",
    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",
    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",
    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",
    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",
    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",
    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",
    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",
    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",
    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slovenia   (SI)",
    "Spain         (ES)", "Sweden         (SE)", "Switzerland      (CH)", "Turkey     (TR)",
    "Ukraine       (UA)", "United Kingdom (UK)"
]

# Set the path where you want to create the folders
base_path = '/home/ray/Dispa-SET_Unleash/Database_PyPSA/Reference_Scenario/HydroData/ScaledInflows'

# Ensure the base path exists
os.makedirs(base_path, exist_ok=True)

# Loop through the list of countries
for country_string in countries:
    # Use a regular expression to find the acronym inside the parentheses
    match = re.search(r'\((.*?)\)', country_string)

    # If a match is found, extract the acronym
    if match:
        acronym = match.group(1).strip()  # Use strip() to remove any extra whitespace
        folder_path = os.path.join(base_path, acronym)

        # Check if the folder already exists to avoid errors
        if not os.path.exists(folder_path):
            os.makedirs(folder_path)
            print(f"Created folder: {folder_path}")
        else:
            print(f"Folder already exists: {folder_path}")
    else:
        print(f"Could not extract acronym from: {country_string}")

print("\nAll folders created successfully!")
'''

<>:1: SyntaxWarning: invalid escape sequence '\('
<>:1: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2404533/2889290989.py:1: SyntaxWarning: invalid escape sequence '\('
  '''


'\n# List of countries with their acronyms in parentheses\ncountries = [\n    "Albania       (AL)", "Armenia        (AM)", "Austria          (AT)", "Azerbaijan (AZ)",\n    "Belarus       (BY)", "Belgium        (BE)", "Bosnia and Herz. (BA)", "Bulgaria   (BG)",\n    "Croatia       (HR)", "Cyprus         (CY)", "Czech Republic   (CZ)", "Denmark    (DK)",\n    "Estonia       (EE)", "Finland        (FI)", "France           (FR)", "Georgia    (GE)",\n    "Germany       (DE)", "Greece         (EL)", "Hungary          (HU)", "Iceland    (IS)",\n    "Ireland       (IE)", "Italy          (IT)", "Kosovo           (XK)", "Latvia     (LV)",\n    "Lithuania     (LT)", "Luxembourg     (LU)", "Malta            (MT)", "Moldova    (MD)",\n    "Montenegro    (ME)", "Netherlands    (NL)", "North Macedonia  (MK)", "Norway     (NO)",\n    "Poland        (PL)", "Portugal       (PT)", "Romania          (RO)", "Russia     (RU)",\n    "Russia Legacy (RU)", "Serbia         (RS)", "Slovakia         (SK)", "Slove

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    2. Dispa-SET_Unleash Folder Path
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Determinning dynamically the zone_folder_path based on the location of the "Dispa-SET_Unleash" folder relative to the current working directory. 
<br>
    If the "Dispa-SET_Unleash" folder is copied to a different machine or location, the dispaSET_unleash_folder_path variable will automatically adjust accordingly.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [3]:
# 1. ------------------------------------------------------------------------------  Get the current working directory 
current_directory = os.getcwd()

# 2. --------------------------------------------------------  Navigate to the parent directory of "Dispa-SET_Unleash" 
dispaSET_unleash_parent_directory = os.path.dirname(current_directory)

# 3. ------------------------------------------------------------------ Get the path to the "Dispa-SET_Unleash" folder 
dispaSET_unleash_folder_path = os.path.dirname(dispaSET_unleash_parent_directory)

# 4. ------------------------------------------------------------- Construct the dispaSET_unleash_folder_name variable 
dispaSET_unleash_folder_name = os.path.basename(dispaSET_unleash_folder_path)

# 5. ------------------------------------------------------------------ ----------------------------------------- Done 
print("dispaSET_unleash_folder_name:", dispaSET_unleash_folder_name)
print("dispaSET_unleash_folder_path:", dispaSET_unleash_folder_path)

dispaSET_unleash_folder_name: Dispa-SET_Unleash
dispaSET_unleash_folder_path: /home/ray/Dispa-SET_Unleash


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.1. PyPSA Source Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
There are two sccenarios as source of PyPSA power plants raw data.
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
Reference_Scenario
<li>
Suficiency_Scenario
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The scenario variable must be selected before proceeding to the next processing steps.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [4]:
# 1. ----------------------------------------------------------------------------------------- Set the needed scenatio 
#pypsa_scenario = "Reference_Scenario"
pypsa_scenario = "Suficiency_Scenario"

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("PyPSA selected Scenario:", pypsa_scenario)

PyPSA selected Scenario: Suficiency_Scenario


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.2. Dispa-SET Time Step
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The time series data must be resampled to a predetermined time step.
<br>
The UNLEASH project utilizes three levels of granularity:
</div>
<div style="text-align: justify; margin-left: 2.0em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman; color:skyblue">
<li>
One hour (1h) 
<li>
Thirty minutes (30min)
<li>
Fifteen minutes (15min)
</li>
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [5]:
# 1. ---------------------------------------------------------------- Set the time step to which data is formating to: 
data_target_time_step = '1h'
# data_target_time_step = '15min'
# data_target_time_step = '30min'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected time step:", data_target_time_step)

Selected time step: 1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
2.3. Secondary Folder Paths
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The many subfolders within the Unleash directory require their location paths to be defined for correct access during processing.
<br>
All of these are dependent on the chosen scenario.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [6]:
# 1. ---------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the data used as base 
additional_path_1 = os.path.join("/Database/CrossBorderFlows", data_target_time_step)
cross_border_flows_base_data_folder_path = dispaSET_unleash_folder_path + additional_path_1

# 2. ---------------- Construct the "Dispa-SET_Unleash_Cross_Border_Flows" name variable of the data used as reference 
cross_border_flows_base_data_folder_name = os.path.basename(cross_border_flows_base_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_base_data_folder_name:", cross_border_flows_base_data_folder_name)
print("cross_border_flows_base_data_folder_path:", cross_border_flows_base_data_folder_path)
# ===========================================================================================================================================

# 1. ------------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the PyPSA raw data 
additional_path_2 = os.path.join("RawData_PyPSA", pypsa_scenario, "CrossBorderFlows")
cross_border_flows_pypsa_raw_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_2

# 2. ------------------- Construct the Dispa-SET_Unleash_Cross_Border_Flows_folder_name variable of the PyPSA raw data 
cross_border_flows_pypsa_raw_data_folder_name = os.path.basename(cross_border_flows_pypsa_raw_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_pypsa_raw_data_folder_name:", cross_border_flows_pypsa_raw_data_folder_name)
print("cross_border_flows_folder_path:", cross_border_flows_pypsa_raw_data_folder_path)
# ===========================================================================================================================================

# 1. -------------------- Get the path to the "Dispa-SET_Unleash_Cross_Border_Flows" folder of the PyPSA formated data 
additional_path_3 = os.path.join("Database_PyPSA", pypsa_scenario, "CrossBorderFlows", data_target_time_step)
cross_border_flows_pypsa_formated_data_folder_path = dispaSET_unleash_folder_path + '/' + additional_path_3

# 2. ------------ Construct the "Dispa-SET_Unleash_Cross_Border_Flows" folder name variable of the PyPSA formated data 
cross_border_flows_pypsa_formated_data_folder_name = os.path.basename(cross_border_flows_pypsa_formated_data_folder_path)

# 3. ------------------------------------------------------------------------------------------------------------ Done
print("cross_border_flows_pypsa_formated_data_folder_name:", cross_border_flows_pypsa_formated_data_folder_name)
print("cross_border_flows_pypsa_formated_data_folder_path:", cross_border_flows_pypsa_formated_data_folder_path)

cross_border_flows_base_data_folder_name: 1h
cross_border_flows_base_data_folder_path: /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h
cross_border_flows_pypsa_raw_data_folder_name: CrossBorderFlows
cross_border_flows_folder_path: /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/CrossBorderFlows
cross_border_flows_pypsa_formated_data_folder_name: 1h
cross_border_flows_pypsa_formated_data_folder_path: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/CrossBorderFlows/1h


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    3. Zone(s) Creation
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Entering the zone name or names (comment those ones that are not available data) where all data related to the corresponding zone are going to be storage.
<br>
For European country names use the ISO 3166-1 standars i.e. AT, BE, BG, CH.... etc. to give the zone_name.
</div>
<hr style="border: 1px solid skyblue;">

In [7]:
# 1. -------------------------------------------------------------------- Set the list of folder names to be addressed 
zone_names = [
                #"AL",
                #"AM",
                #"AT",
                #"AZ",
                #"BY",
                "BE",
                #"BA",
                #"BG",
                #"HR",
                #"CY",
                #"CZ",
                #"DK",
                #"EE",
                #"FI",
                "FR",
                #"GE",
                "DE",
                #"EL",
                #"HU",
                #"IS",
                #"IE",
                #"IT",
                #"XK",
                #"LV",
                #"LT",
                #"LU",
                #"MT",
                #"MD",
                #"ME",
                "NL",
                #"MK",
                #"NO",
                #"PL",
                #"PT",
                #"RO",
                #"RU",
                #"RS",
                #"SK",
                #"SI",
                #"ES",
                #"SE",
                #"CH",
                #"TR",
                #"UA",
                "UK"
             ]

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected zones:", zone_names)

Selected zones: ['BE', 'FR', 'DE', 'NL', 'UK']


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The following dictionary specifies the possible alternative names—or synonyms/aliases—used in international nomenclature for the EU countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [8]:
# 1. --------------------------------------------------------- Set the list of the alternative acronyms per EU country 
zone_names_equivalences_dict = {

"AL"  :  {"Acronym": ["  "       ] ,   "name": ["Albania   "       ]},  
"AM"  :  {"Acronym": ["  "       ] ,   "name": ["Armenia   "       ]},
"AT"  :  {"Acronym": ["  "       ] ,   "name": ["Austria   "       ]},
"AZ"  :  {"Acronym": ["  "       ] ,   "name": ["Azerbaijan"       ]},
"BY"  :  {"Acronym": ["  "       ] ,   "name": ["Belarus"          ]},
"BE"  :  {"Acronym": ["  "       ] ,   "name": ["Belgium"          ]},
"BA"  :  {"Acronym": ["  "       ] ,   "name": ["Bosnia and Herz." ]},
"BG"  :  {"Acronym": ["  "       ] ,   "name": ["Bulgaria"         ]},
"HR"  :  {"Acronym": ["  "       ] ,   "name": ["Croatia"          ]},
"CY"  :  {"Acronym": ["  "       ] ,   "name": ["Cyprus"           ]},
"CZ"  :  {"Acronym": ["  "       ] ,   "name": ["Czech Republic"   ]},
"DK"  :  {"Acronym": ["  "       ] ,   "name": ["Denmark"          ]},
"EE"  :  {"Acronym": ["  "       ] ,   "name": ["Estonia"          ]},
"FI"  :  {"Acronym": ["  "       ] ,   "name": ["Finland"          ]},
"FR"  :  {"Acronym": ["  "       ] ,   "name": ["France"           ]},
"GE"  :  {"Acronym": ["  "       ] ,   "name": ["Georgia"          ]}, 
"DE"  :  {"Acronym": ["  "       ] ,   "name": ["Germany"          ]},
"EL"  :  {"Acronym": ["GR"       ] ,   "name": ["Greece"           ]},
"HU"  :  {"Acronym": ["  "       ] ,   "name": ["Hungary"          ]},
"IS"  :  {"Acronym": ["  "       ] ,   "name": ["Iceland"          ]},
"IE"  :  {"Acronym": ["  "       ] ,   "name": ["Ireland"          ]},
"IT"  :  {"Acronym": ["  "       ] ,   "name": ["Italy"            ]},
"XK"  :  {"Acronym": ["  "       ] ,   "name": ["Kosovo"           ]},
"LV"  :  {"Acronym": ["  "       ] ,   "name": ["Latvia"           ]},
"LT"  :  {"Acronym": ["  "       ] ,   "name": ["Lithuania"        ]},
"LU"  :  {"Acronym": ["  "       ] ,   "name": ["Luxembourg"       ]},
"MT"  :  {"Acronym": ["  "       ] ,   "name": ["Malta"            ]},
"MD"  :  {"Acronym": ["  "       ] ,   "name": ["Moldova"          ]},
"ME"  :  {"Acronym": ["  "       ] ,   "name": ["Montenegro"       ]},
"NL"  :  {"Acronym": ["  "       ] ,   "name": ["Netherlands"      ]},
"MK"  :  {"Acronym": ["  "       ] ,   "name": ["North Macedonia"  ]},
"NO"  :  {"Acronym": ["  "       ] ,   "name": ["Norway"           ]},
"PL"  :  {"Acronym": ["  "       ] ,   "name": ["Poland"           ]},
"PT"  :  {"Acronym": ["  "       ] ,   "name": ["Portugal"         ]},
"RO"  :  {"Acronym": ["  "       ] ,   "name": ["Romania"          ]},
"RU"  :  {"Acronym": ["  "       ] ,   "name": ["Russia"           ]},
"RS"  :  {"Acronym": ["  "       ] ,   "name": ["Serbia"           ]},
"SK"  :  {"Acronym": ["  "       ] ,   "name": ["Slovakia"         ]},
"SI"  :  {"Acronym": ["  "       ] ,   "name": ["Slovenia"         ]},
"ES"  :  {"Acronym": ["  "       ] ,   "name": ["Spain"            ]},
"SE"  :  {"Acronym": ["  "       ] ,   "name": ["Sweden"           ]},
"CH"  :  {"Acronym": ["  "       ] ,   "name": ["Switzerland"      ]},
"TR"  :  {"Acronym": ["  "       ] ,   "name": ["Turkey"           ]},
"UA"  :  {"Acronym": ["  "       ] ,   "name": ["Ukraine"          ]},
"UK"  :  {"Acronym": ["GB"       ] ,   "name": ["United Kingdom"   ]},
       
}

# 2. ------------------------------------------------ Create a new dictionary with only the keys present in zone_names
selected_zone_names_equivalences_dict = {
    key: zone_names_equivalences_dict[key]
    for key in zone_names
    if key in zone_names_equivalences_dict
}

# 3. ------------------------------------------------------------------------------------------------------------ Done 
for key, value in selected_zone_names_equivalences_dict.items():
    if any(acronym.strip() for acronym in value["Acronym"]):
        print(f"Key: {key};    Acronym: {value['Acronym']};    Name: {value['name']}\n")

Key: UK;    Acronym: ['GB'];    Name: ['United Kingdom']



<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
    4. Data Reference Year 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    Setting the variable on the target year which formatting data is wanted to
</div>
<hr style="border: 1px solid skyblue;">

In [9]:
# 1. --------------------------------------------------------------------- Set the year to which data is formating to: 
data_target_year = '2050'

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print("Selected Year:", data_target_year)

Selected Year: 2050


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [10]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_name}\n")
print (f"Path to the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):           {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Target time step:                                          {data_target_time_step}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Cross Border Flows Base data folder:           1h

Path to the Cross Border Flows Base data folder:           /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h

Name of the Cross Border Flows_Pypsa Raw data 1 folder:    CrossBorderFlows

Path to the Cross Border Flows_Pypsa Raw data 1 folder:    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/CrossBorderFlows

Name of the Cross Border Flows_Pypsa Formated data folder: 1h

Path to the Cross Border Flows_Pypsa Formated data folder: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/CrossBorderFlows/1h

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dictionary):           [['  '], ['  '], ['  '], ['  '], ['GB']]

Target year:                            

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
4. Cross Border FLows Data Frame
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating the data frame with all the corresponding headers according the Dispa-SET nomenclature.
</div>
<div style="text-align: justify; margin-left: 4.5em; font-weight: bold; font-size: 15px; font-family: TimesNewRoman; color:skyblue">
Sources:
</div>
<div style="text-align: justify; margin-left: 5em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
    <ol style="margin-top: 0; padding-left: 1.5em;">
        <li>
The Dispa-SET documentation.
            <br>
            <a href="https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81" style="color:skyblue">https://www.dispaset.eu/en/latest/data.html#:~:text=Interconnections%EF%83%81</a>
        </li>
</div>
<hr style="border: 2px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.1. Empty Zone Data Frames
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Creating empty dataframes—column headers only—for the selected zones, corresponding to the target year.
<br>
Loading the csv bade file.
</div>
<hr style="border: 1px solid skyblue;">
</div>

In [11]:
# 1. ------------------------------------------------------------------------------------ Convert data year to integer 
data_target_year = int(data_target_year)  

# 2. ----------------------------------------------------------------------------- Find the most appropriate csv files
csv_files = [
    f for f in os.listdir(cross_border_flows_base_data_folder_path)
    if f.lower().endswith(".csv")
]

# 3. --------------------------------------------------------------------- Keep only filenames that are exactly a year
available_years = {
    int(os.path.splitext(f)[0]): f
    for f in csv_files
    if os.path.splitext(f)[0].isdigit()
}

if not available_years:
    raise ValueError("No CSV files named with a year found")

# 4. ------------------------------------------------------------------------------- Select exact year or closest year
if data_target_year in available_years:
    selected_year = data_target_year
else:
    selected_year = min(
        available_years.keys(),
        key=lambda y: abs(y - data_target_year)
    )

selected_csv_path = os.path.join(
    cross_border_flows_base_data_folder_path,
    available_years[selected_year]
)

# 5. ----------------------------------------------------------------------------------- Load csv file into data frame
cross_border_flows_df = pd.read_csv(selected_csv_path)

# 6. ------------------------------------------------------------------------------------------------------------ Done
print(f"CSV year used: {selected_year}")
print(cross_border_flows_df.columns.tolist())
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.")
cross_border_flows_df

CSV year used: 2024
['index', 'BG -> RoW', 'RoW -> BG', 'EE -> RoW', 'RoW -> EE', 'EL -> RoW', 'RoW -> EL', 'FI -> RoW', 'RoW -> FI', 'HR -> RoW', 'RoW -> HR', 'HU -> RoW', 'RoW -> HU', 'IT -> RoW', 'RoW -> IT', 'LT -> RoW', 'RoW -> LT', 'LV -> RoW', 'RoW -> LV', 'PL -> RoW', 'RoW -> PL', 'RO -> RoW', 'RoW -> RO', 'SK -> RoW', 'RoW -> SK']
✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.


,index,BG -> RoW,RoW -> BG,EE -> RoW,RoW -> EE,EL -> RoW,RoW -> EL,FI -> RoW,RoW -> FI,HR -> RoW,...,LT -> RoW,RoW -> LT,LV -> RoW,RoW -> LV,PL -> RoW,RoW -> PL,RO -> RoW,RoW -> RO,SK -> RoW,RoW -> SK
0,2024-01-01 00:00:00+00:00,827.0,0.0,0.0,109.0,0.0,1129.0,0.0,0.0,534.0,...,0.0,68.0,178.0,0.0,0.0,0.0,159.0,367.0,340.0,0.0
1,2024-01-01 01:00:00+00:00,852.0,0.0,0.0,83.0,0.0,1170.0,0.0,0.0,466.0,...,0.0,72.0,178.0,0.0,0.0,0.0,223.0,283.0,247.0,0.0
2,2024-01-01 02:00:00+00:00,823.0,0.0,0.0,255.0,0.0,1158.0,0.0,0.0,334.0,...,43.0,0.0,220.0,0.0,0.0,0.0,302.0,204.0,197.0,0.0
3,2024-01-01 03:00:00+00:00,808.0,0.0,0.0,174.0,0.0,1178.0,0.0,0.0,184.0,...,0.0,20.0,192.0,0.0,0.0,0.0,425.0,159.0,138.0,0.0
4,2024-01-01 04:00:00+00:00,675.0,0.0,0.0,156.0,1.0,1174.0,0.0,0.0,77.0,...,0.0,47.0,190.0,0.0,0.0,0.0,512.0,154.0,140.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,42.0,191.0,0.0,51.0,279.0,58.0,0.0,0.0,1048.0,...,95.0,0.0,71.0,0.0,425.0,0.0,273.0,821.0,965.0,0.0
8780,2024-12-31 20:00:00+00:00,131.0,145.0,0.0,81.0,387.0,79.0,0.0,0.0,923.0,...,107.0,0.0,59.0,0.0,397.0,0.0,136.0,624.0,935.0,0.0
8781,2024-12-31 21:00:00+00:00,227.0,120.0,0.0,163.0,412.0,144.0,0.0,0.0,1129.0,...,77.0,0.0,50.0,0.0,445.0,0.0,238.0,638.0,965.0,0.0
8782,2024-12-31 22:00:00+00:00,343.0,46.0,0.0,213.0,278.0,212.0,0.0,0.0,1326.0,...,133.0,0.0,25.0,0.0,366.0,0.0,279.0,699.0,976.0,0.0


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Identifying the interconnections between the selected zones/countries.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [12]:
# 1. ------------------------------------------------------------------------- Assuming zone_names is the original list
zone_names_with_row = zone_names + ['RoW']

# 2. ------------------------------------------------ Build all ordered zone combinations (A -> B), now including 'RoW'
zone_combinations = [f"{a} -> {b}" for a, b in permutations(zone_names_with_row, 2)]
zone_combinations_set = set(zone_combinations)

# 3. ----------------------------------------------------------- Identify columns that look like a combination (X -> Y)
combination_like_cols = [
    col for col in cross_border_flows_df.columns
    if isinstance(col, str) and " -> " in col
]

# 4. ------------------------------------------------------ Among the combination-like columns, keep only exact matches
matching_cols = [col for col in combination_like_cols if col in zone_combinations_set]

# 5. ---------------------------------------------------------------- Remove combination-like columns that do NOT match
cols_to_drop = set(combination_like_cols) - set(matching_cols)
cross_border_flows_df = cross_border_flows_df.drop(columns=cols_to_drop)

# 6. --------------------------------------------------------------------- Create missing combinations as empty columns
missing_combinations = zone_combinations_set - set(matching_cols)
for col in missing_combinations:
    cross_border_flows_df[col] = np.nan

# 7. --------------------------------------- Sort columns if desired: keep original order + new combinations at the end
cross_border_flows_df = cross_border_flows_df[list(cross_border_flows_df.columns)]

# 8. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.")
cross_border_flows_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_df'.


,index,FR -> RoW,FR -> NL,DE -> FR,DE -> RoW,BE -> RoW,FR -> BE,FR -> UK,NL -> UK,UK -> RoW,...,UK -> NL,UK -> BE,RoW -> DE,RoW -> NL,NL -> RoW,NL -> DE,DE -> UK,RoW -> FR,UK -> DE,FR -> DE
0,2024-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8780,2024-12-31 20:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8781,2024-12-31 21:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8782,2024-12-31 22:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
4.2. Time Step Correction
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Verifying and correcting first column timestamps in the dataframe to match target year and selected time step.
<hr style="border: 1px solid skyblue;">
</div>

In [13]:
# 1. ---------------------------------------------- Define a mapping for time step strings to pandas frequency strings
time_step_map = {
    "1h"   : "H",
    "15min": "15T",
    "30min": "30T"
}

# 2. -------------------------------------------------------- Get the pandas frequency string for the target time step
freq = time_step_map[data_target_time_step]

# Assume `df` is your single DataFrame
if cross_border_flows_df.empty:
    print("DataFrame is empty. Skipping alignment.")
else:
    
# 2.1 -------------------------------------------------------------------------------------- Identify the first column
    first_col = cross_border_flows_df.columns[0]

# 2.2 ------------------------------------------------------------- Convert column to datetime with UTC if not already
    cross_border_flows_df[first_col] = pd.to_datetime(cross_border_flows_df[first_col], utc=True, errors='coerce')

# 2.3 --------------------------------------------------------------------------- Remove any rows that failed to parse
    cross_border_flows_df = cross_border_flows_df.dropna(subset=[first_col])

# 2.4 --------------------------------------------------------- Create a date range for the correct year and time step
    start_time = pd.Timestamp(f"{data_target_year}-01-01 00:00:00", tz="UTC")
    end_time = pd.Timestamp(f"{data_target_year}-12-31 23:59:59", tz="UTC")

    correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)

# 2.5 --------------------------------------------------------- Replace the first column with the corrected date range
    if len(correct_index) >= len(cross_border_flows_df):
        cross_border_flows_df[first_col] = correct_index[:len(cross_border_flows_df)]
    else:
        
# 2.5.1 ----------------------------------------- If df has more rows than the date range, extend with repeated values
        repeats = (len(cross_border_flows_df) // len(correct_index)) + 1
        cross_border_flows_df[first_col] = pd.Series(list(correct_index) * repeats)[:len(cross_border_flows_df)]

# 2.6 ----------------------------------------------------------------------------------------------------------- Done
    print(f"Updated DataFrame: first column aligned with {data_target_year} and {data_target_time_step}")

Updated DataFrame: first column aligned with 2050 and 1h


/tmp/ipykernel_2404533/1556487799.py:29: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  correct_index = pd.date_range(start=start_time, end=end_time, freq=freq)


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
<div style="text-align: right; margin-left: 3.0em; font-weight: bold; font-size: 14px; font-family: TimesNewRoman;color:skyblue">
    Tracking Variables.
</div>
    <div style="text-align: right; margin-left: 1.50em; font-weight: unbold; font-size: 13px; font-family: TimesNewRoman;color:skyblue">
    This cells are just to confirm all the file names, file paths and other information related to the data being processed.
    <br>
  Also are used to ensure the inputs for next cells in order to avoid to re-enter the same information each time.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [14]:
print (f"Name of the DispaSET Unleash folder:                       {dispaSET_unleash_folder_name}\n")
print (f"Path to the DispaSET Unleash folder:                       {dispaSET_unleash_folder_path}\n")
print (f"Name of the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_name}\n")
print (f"Path to the Cross Border Flows Base data folder:           {cross_border_flows_base_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Raw data 1 folder:    {cross_border_flows_pypsa_raw_data_folder_path}\n")
print (f"Name of the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_name}\n")
print (f"Path to the Cross Border Flows_Pypsa Formated data folder: {cross_border_flows_pypsa_formated_data_folder_path}\n")
print (f"Name of the zones:                                         {zone_names}\n")
print (f"Name of the zone_names_equivalences_dictionary):           {[value['Acronym'] for value in selected_zone_names_equivalences_dict.values()]}\n")
print (f"Target year:                                               {data_target_year}\n")
print (f"Target time step:                                          {data_target_time_step}\n")
print (f"Name of the Cross Border Flows DataFrame:                  {list(cross_border_flows_df.keys())}\n")

Name of the DispaSET Unleash folder:                       Dispa-SET_Unleash

Path to the DispaSET Unleash folder:                       /home/ray/Dispa-SET_Unleash

Name of the Cross Border Flows Base data folder:           1h

Path to the Cross Border Flows Base data folder:           /home/ray/Dispa-SET_Unleash/Database/CrossBorderFlows/1h

Name of the Cross Border Flows_Pypsa Raw data 1 folder:    CrossBorderFlows

Path to the Cross Border Flows_Pypsa Raw data 1 folder:    /home/ray/Dispa-SET_Unleash/RawData_PyPSA/Suficiency_Scenario/CrossBorderFlows

Name of the Cross Border Flows_Pypsa Formated data folder: 1h

Path to the Cross Border Flows_Pypsa Formated data folder: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/CrossBorderFlows/1h

Name of the zones:                                         ['BE', 'FR', 'DE', 'NL', 'UK']

Name of the zone_names_equivalences_dictionary):           [['  '], ['  '], ['  '], ['  '], ['GB']]

Target year:                            

<div style="background-color: black;">
<hr style="border: 2px solid skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: 'Times New Roman', serif; color: skyblue;">
5. PyPSA vs Dispaset Nomenclature
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
All those technologies from PyPSA which can be represented in Dispa-SET have to be identified.
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.1. PyPSA Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
The next chart represents graphically how all the Energy sector inside PyPSA is structured.<br>
This is used to get the equivalent diagram for Dispaset.
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_multisector_figure_1.png" 
       alt="PyPSA Multisector Flow Diagram" 
       style="max-width:35%; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: <a href="https://pypsa-eur.readthedocs.io/en/latest/" target="_blank" style="color: skyblue; text-decoration: underline;">PyPSA-Eur Documentation</a>
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.2. Equivalent Dispa-SET Energy System Flow Diagram
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
A correlation has been established between the PyPSA parameters and their corresponding Dispa-SET equivalents:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color:skyblue">
Due to feature limitations, not all elements from PyPSA can be represented in Dispa-SET.
<br>
However, for those compatible technologies, the following chart graphically illustrates how they are connected within the Dispa-SET environment logic:
</div>
<div style="text-align: center; margin: 20px 0;">
  <img src="Images/PyPSA_sectors_as_Dispaset_Flow_work_Filtered_1.svg" 
       alt="Equivalent Dispa-SET Energy System Flow Diagram" 
       style="max-width: auto; height: auto;">
</div>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 2.0px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Technologies, Demmands & Interconnection Lines Nomenclature - PyPSA vs Dispaset 
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
The PyPSA technologies and demmands nomenclature and their correlation with their homologous from Dispa-SET are described as follows:
</div>
<table style="width: 95%; margin-left: auto; margin-right: auto; border-collapse: collapse; font-family: TimesNewRoman; font-size: 12px; color: skyblue;">
  <thead>
    <tr style="background-color: #1E1E1E; color: skyblue; border-bottom: 1px solid skyblue;">
      <th style="width: 8%; padding: 8px; text-align: left; border: 1px solid #444;">PyPSA Element</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Technology</th>
      <th style="width: 10%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Classification</th>
      <th style="width: 34%; padding: 8px; text-align: left; border: 1px solid #444;">Description</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Sector Relation</th>
      <th style="width: 15%; padding: 8px; text-align: left; border: 1px solid #444;">Dispa-SET Element</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">Element Type</th>
      <th style="width: 5%; padding: 8px; text-align: left; border: 1px solid #444;">May Modeled?</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DC</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents the DC (HVDC) transmission network for electricity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">NTC</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">OCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Open-Cycle Gas Turbine producing electricity from gas.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CCGT</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined-Cycle Gas Turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">EV charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Interface between the grid and electric vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">V2G</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Vehicle-to-Grid---allows EVs to discharge electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to stored energy in batteries (charging link)</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxChargingPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">BioSNG</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces synthetic natural gas (bio-methane)---fuel synthesis process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">DAC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct Air Capture---captures CO<sub>2</sub> for storage or utilization</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Fischer-Tropsch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + CO<sub>2</sub> to liquid hydrocarbons; fuel synthesis</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Electrolysis</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity into hydrogen cross-sector conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> Fuel Cell</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts hydrogen back to electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports hydrogen between regions or sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> pipeline retrofitted</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Existing pipelines adapted for H<sub>2</sub> transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> turbine</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Generates electricity or heat from hydrogen---boundary technology</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Haber-Bosch</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts H<sub>2</sub> + N<sub>2</sub> into ammonia---chemical/fertilizer industry</td>
      <td style="padding: 8px; border: 1px solid #444;">PX2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Steam Methane Reforming---gas to hydrogen conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">SMR with Carbon Capture---industrial hydrogen with CC</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">Sabatier</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub> + CO<sub>2</sub> → CH<sub>4</sub>---synthetic methane production.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture machinery oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil used in agricultural machinery---transport/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">ammonia cracker</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts ammonia back into hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">battery discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored electricity from batteries</td>
      <td style="padding: 8px; border: 1px solid #444;">P&S</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity distribution grid</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents distribution-level power flow---low voltage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents coal-based electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">nuclear</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear-to-electricity conversion within the power system.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation or conversion node</td>
      <td style="padding: 8px; border: 1px solid #444;">X2P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Upgrades raw biogas into pipeline-quality methane</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas to gas CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biogas upgrading with carbon capture---industrial fuel conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">biomass to liquid</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass into liquid fuels---synthetic fuel process.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents captured Tons of CO<sub>2</sub> / hour transported or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">coal for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Use of coal as industrial feedstock/fuel</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Supplies natural gas to industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial gas use with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transports natural gas---energy carrier infrastructure</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">gas pipeline new</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Expansion of natural gas transport capacity</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Boundary Sector Interconnections</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">kerosene for aviation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Aviation fuel consumption---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil use for land transport---transport fuel consumption</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">methanolisation</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Synthesizes methanol (CO<sub>2</sub> + H<sub>2</sub> → CH<sub>3</sub>OH)</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides naphtha feedstock to industrial processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial process CO<sub>2</sub> emissions. Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">process emissions CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Industrial CO<sub>2</sub> emissions with capture — Tons of CO<sub>2</sub> / hour</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for rural homes</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts biomass to heat---residential fuel use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---non-electric final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Uses electricity directly for heating---part of demand side</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to thermal energy in storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat — part of residential heating loop</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized air-source heat pump for urban buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating devices---boundary heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass-to-heat conversion for urban residential areas</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban gas boilers---distributed thermal devices</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized oil boilers for urban homes</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">link & residential rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heat using stable ground temperature</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
     <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Transfers electric energy to thermal storage (heat)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges heat from thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Service sector rural building heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Provides space or process heat for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns gas for heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural ground heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity-to-heat for service buildings---heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating in rural service buildings.</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric heating for service buildings---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat for service‐sector thermal storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to buildings---part of the heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Local electric heat production for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral biomass boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass‐to‐heat conversion---end-use heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Burns oil for household heating---outside power generation</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral oil boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil heating---final energy use</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Direct electric heating for service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts power to stored heat</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Releases stored thermal energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Methanol use in maritime transport---fuel demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption for ships---transport sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass fuel use for industrial heat/processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for industry CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same as above but with carbon capture</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerXx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass transport</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents biomass logistics between regions/sectors</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">FlowXmaximum & FlowXminimum</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central air heat pump</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized district heat pump---heat sector interfac.</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Combined heat + power supplying district heating</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Gas CHP with carbon capture---district heating system</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Plant Data</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central gas boiler</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized gas heating for urban networks</td>
      <td style="padding: 8px; border: 1px solid #444;">X2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central resistive heater</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Electric boiler for district heating---end-use conversion</td>
      <td style="padding: 8px; border: 1px solid #444;">P2X</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass combined heat + power---heat boundary process</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solid biomass CHP CC</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Same with carbon capture---boundary sector</td>
      <td style="padding: 8px; border: 1px solid #444;">X2CHP</td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks charger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Converts electricity to heat in district storage</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">link</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks discharger</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Discharges stored heat to the district network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputSectorXStorageInput</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fossil fuel-based electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas–fired power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal variant used for power generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
        <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil-fired electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Powerx2p</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">onwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Onshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind turbine generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-ac</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with AC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">offwind-dc</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Offshore wind with DC connection</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Utility-scale photovoltaic generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">solar rooftop</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed PV connected to power grid</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">ror</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Run-of-river hydro power plant</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel input for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputPower</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Produces heat for households (not electricity)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized solar heating for buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Solar thermal for service-sector heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Urban service-sector solar heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central solar thermal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Centralized solar thermal for district heating</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity & STOMaxPower</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">generator</td>
      <td style="padding: 8px; border: 1px solid #444;">load</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Represents total shredding energy</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Load Shedding</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">hydro</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional hydro reservoir</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">storage_units</td>
      <td style="padding: 8px; border: 1px solid #444;">PHS</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Pumped Hydro Storage, a grid-scale electricity storage technology</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">PowerCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">battery</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electrical energy storage — directly coupled with the grid.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
            <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">uranium</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Nuclear fuel stock for electricity generation</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">H<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen storage---chemical energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">NH<sub>3</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Ammonia storage---chemical/fertilizer or fuel vector.</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">biogas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomethane stock for heating or industry</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Captured CO<sub>2</sub> pool---used in synthesis or stored</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> sequestered</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Permanent CO<sub>2</sub> storage---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>      
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">CO<sub>2</sub> stored</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Intermediate or final CO<sub>2</sub> reservoir---Tons of CO<sub>2</sub></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">OutputEmissions</td>
      <td style="padding: 8px; border: 1px solid #444;">output</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">coal</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Coal stock for industrial/fuel processes</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">gas</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas (CH<sub>4</sub>) stock — cross-sector energy carrier</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">lignite</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Fuel storage for thermal use — outside grid operations</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">methanol</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Liquid fuel stock — used in transport or synthesis chains</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil (synthetic hydrocarbons) stock for transport/industrial fuels</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass stock for heating/industrial use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for rural households — heat sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1评审44;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">residential urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Distributed heat storage in urban residences</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services rural water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat storage for rural service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">services urban decentral water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Thermal storage for urban service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">stores</td>
      <td style="padding: 8px; border: 1px solid #444;">urban central water tanks</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating storage — boundary heat network</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">STOCapacity</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Final oil demand in the industrial sector</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary DH demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">District heating demand for residential and service buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Residential and tertiary heat demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Decentralized residential space/water heating demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in lighting, irrigation, machinery)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture heat</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Heat energy required in agricultural processes (drying, greenhouses)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">agriculture oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil consumption in agricultural machinery and vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">aviation oil demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Jet fuel (kerosene) demand for aviation transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand for rail network</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Traction electricity used by rail and metro transport systems</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity demand of residential and tertairy</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption in households and service-sector buildings</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">electricity for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity use for machinery, processes, and electrified production</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">methane</td>
      <td style="padding: 8px; border: 1px solid #444;">gas for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Natural gas or synthetic methane Industrial demand</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">hydrogen for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand for industrial refining, ammonia, steelmaking</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">electricity</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport EV</td>
      <td style="padding: 8px; border: 1px solid #444;">Power Sector</td>
      <td style="padding: 8px; border: 1px solid #444;">Electricity consumption by electric vehicles in road transport</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">Demand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">land transport hydrogen demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen fuel demand for road transport (fuel-cell vehicles)</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">heat</td>
      <td style="padding: 8px; border: 1px solid #444;">low-temperature heat for industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">(below $\sim 200^{\circ}$C), typically supplied by boilers or heat pumps</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">Non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">naphtha for non-energy</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Naphtha used as a chemical feedstock e.g., plastics, petrochemicals</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">N & N</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">oil to transport demand</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Oil demand for conventional land gasoline and diesel vehicles</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping hydrogen</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Hydrogen demand, maritime transport fuel-cell/ combustion ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #4D4D4D;">
      <td style="padding: 8px; border: 1px solid #444;">oil</td>
      <td style="padding: 8px; border: 1px solid #444;">shipping oil</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Conventional marine oil fuel demand (HFO, MGO) for ships</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
    <tr style="background-color: #333333;">
      <td style="padding: 8px; border: 1px solid #444;">solid biomass</td>
      <td style="padding: 8px; border: 1px solid #444;">solid biomass for Industry</td>
      <td style="padding: 8px; border: 1px solid #444;">Sector X</td>
      <td style="padding: 8px; border: 1px solid #444;">Biomass demand in industrial processes for heat or material use</td>
      <td style="padding: 8px; border: 1px solid #444;"></td>
      <td style="padding: 8px; border: 1px solid #444;">SectorXDemand</td>
      <td style="padding: 8px; border: 1px solid #444;">input</td>
      <td style="padding: 8px; border: 1px solid #444;">Y & Y</td>
    </tr>
  </tbody>
</table>
<div style="text-align: center; margin-top: 10px; font-size: 12px; font-family: 'Times New Roman', serif; color: skyblue;">
  Source: Adapted for Dispa-SET_Unleash
</div>
<hr style="border: 0.5px dashed skyblue;">
<div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: 'Times New Roman', serif; color: skyblue;">
5.3. Dispa-SET vs PyPSA Interconnection Lines Equivalences Dictionary
</div>
<div style="text-align: justify; margin-left: 0.0em; font-weight: normal; font-size: 14px; font-family: 'Times New Roman', serif; color: skyblue;">
To facilitate data harmonization, a dictionary containing the correlations between the raw sources files and the Dispa-SET and PyPSA nomenclature equivalences for interconnection lines is developed, leveraging the specifications detailed in the preceding table.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [15]:
# 1. ------------------------------------------------------- Dictionary mapping Dispa-SET acronyms to PyPSA tech names 
tech_equivalences_dict = {
    
                            # -----------------------
                            # Interconection lines
                            # -----------------------
                            "Tech_AC"  :  {"transmission_lines": ["AC"  ] ,   "capacities": ["AC Transmission lines"  ],   "RM_Factor": ["0.898"  ] } ,
    
                            "Tech_DC"  :  {"transmission_lines": ["DC"  ] ,   "capacities": ["DC Transmission lines"  ],   "RM_Factor": ["1.000"  ] } ,
       
                          }

# 2. ------------------------------------------------------------------------------------------------------------ Done 
print(tech_equivalences_dict)

{'Tech_AC': {'transmission_lines': ['AC'], 'capacities': ['AC Transmission lines'], 'RM_Factor': ['0.898']}, 'Tech_DC': {'transmission_lines': ['DC'], 'capacities': ['DC Transmission lines'], 'RM_Factor': ['1.000']}}


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 18px; font-family: TimesNewRoman; color:skyblue">
        6. PyPSA to Dispa-SET Country Interconnections Flows Data Formatting
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Formatting the raw cross-border flow DataFrame.
    </div>
    <hr style="border: 1px dashed skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
6.1. Interconnection Data Sources
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Uploading the raw data into data frames for the formating process.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [16]:
# 1. ------------------------------------------------------------------------------------------- Define inner variables
folder_path = cross_border_flows_pypsa_raw_data_folder_path

# 2. -------------------------------------------------------- Identify files that contain the target year in their name
files_to_read = [
    f for f in os.listdir(folder_path) 
    if f.endswith('.csv') and str(data_target_year) in f
]

#  2.1. ----------------------------------------------------------------- Sort files to ensure consistent merging order
files_to_read.sort()

# 3. ------------------------------------------------------------------------------------ Process and combine the files
cross_border_flows_raw_data_df = pd.DataFrame()

for i, file_name in enumerate(files_to_read):
    file_full_path = os.path.join(folder_path, file_name)
    temp_df = pd.read_csv(file_full_path)
    
    if cross_border_flows_raw_data_df.empty:
        
# 3.1. ------------------------------------------------------------------------------ First file: just load it entirely
        cross_border_flows_raw_data_df = temp_df
    else:

# 3.1.1. ------------------------------ For subsequent files; Identify columns that already exist in the main DataFrame
        common_cols = temp_df.columns.intersection(cross_border_flows_raw_data_df.columns)
        
# 3.1.2. ------------------------------------------------- Identify which of these common columns have identical values
        cols_to_drop = []
        for col in common_cols:
            if temp_df[col].equals(cross_border_flows_raw_data_df[col]):
                cols_to_drop.append(col)
        
# 3.1.3. ---------------------------------------------- Drop the identical columns from the current file before joining
        temp_df_filtered = temp_df.drop(columns=cols_to_drop)
        
# 3.1.4. ---------------------------------------------------------------------------- Concatenate horizontally (axis=1)
        cross_border_flows_raw_data_df = pd.concat([cross_border_flows_raw_data_df, temp_df_filtered], axis=1)

# 4. ------------------------------------------------------------------------------------------------------------ Done 
print(f"Successfully loaded {len(files_to_read)} files for year {data_target_year}.")
print(f"Final DataFrame shape: {cross_border_flows_raw_data_df.shape}")
cross_border_flows_raw_data_df

Successfully loaded 2 files for year 2050.
Final DataFrame shape: (8760, 17)


,snapshot,AC_BE1 0-FR1 0,AC_BE1 0-NL1 0,AC_DE1 0-FR1 0,AC_DE1 0-NL1 0,DC_DE1 0-BE1 0,DC_GB0 0-NL1 0,DC_GB0 0-FR1 0,DC_GB0 0-GB2 0,DC_GB0 0-GB2 0.1,DC_GB0 0-FR1 0.1,DC_GB0 0-FR1 0.2,DC_FR1 0-GB0 0,DC_GB0 0-FR1 0.3,DC_GB0 0-DE1 0,DC_GB0 0-NL1 0.1,DC_GB0 0-BE1 0
0,2013-01-01 00:00:00,-104.107714,-3920.604294,-72.693467,-4735.092357,0.003487,2460.370002,4920.655273,0.003112,0.003286,3444.273289,3645.001102,0.001359,3444.333691,3444.519089,2952.444930,2460.370959
1,2013-01-01 01:00:00,-248.143043,-3914.429679,-173.266592,-4722.658899,0.003448,2460.370012,4920.655296,0.003206,0.003379,3444.273312,3645.001124,0.001353,3444.333714,3444.519108,2952.444935,2460.370984
2,2013-01-01 02:00:00,-515.167759,-3899.539575,-359.717472,-4705.476232,0.003433,2460.369989,4920.655292,0.003215,0.003374,3444.273310,3645.001122,0.001353,3444.333712,3444.519106,2952.444914,2460.370980
3,2013-01-01 03:00:00,-849.520353,-3891.289627,-593.180284,-4703.412134,0.003380,2460.369936,4920.655283,0.003193,0.003336,3444.273301,3645.001113,0.001358,3444.333703,3444.519105,2952.444859,2460.370965
4,2013-01-01 04:00:00,-649.203756,-3818.788794,-453.308563,-4615.770263,0.003628,2460.370606,4920.655295,0.003223,0.003362,3444.273317,3645.001128,0.001345,3444.333719,3444.519095,2952.445527,2460.371025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1141.823671,-7630.706649,797.282221,-9114.841750,0.002658,2460.370501,4920.655535,0.004600,0.004648,3444.273570,3645.001379,0.001169,3444.333972,3444.519324,2952.445420,2460.371264
8756,2013-12-31 20:00:00,1715.916067,-7723.255226,1198.144114,-9232.381190,0.002634,2460.370485,4920.655522,0.004584,0.004599,3444.273559,3645.001369,0.001174,3444.333961,3444.519321,2952.445404,2460.371252
8757,2013-12-31 21:00:00,2387.612570,-8251.781594,1667.158398,-9889.501646,0.002603,2460.370466,4920.655511,0.004582,0.004584,3444.273550,3645.001359,0.001179,3444.333952,3444.519320,2952.445384,2460.371241
8758,2013-12-31 22:00:00,2457.405473,-7843.638983,1715.891528,-9415.943669,0.002698,2460.370656,4920.655520,0.003625,0.003660,3444.273560,3645.001369,0.001170,3444.333962,3444.519316,2952.445573,2460.371275


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
        6.2. Interconnected Zone Names Armonization
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Applying the DispaSET interconnection format; i.e. AA -> BB.
        <hr style="border: 0.5px solid skyblue;">
</div>

In [17]:
# 1. --------------------------------------------------- Create a reverse mapping for easy lookup: {Acronym: zone_name}
acronym_to_zone = {}
for zone in zone_names:
    acronyms = zone_names_equivalences_dict[zone].get('Acronym', [])
    
# 1.1. -------------------------------------------------------------------------------------- Ensure acronyms is a list
    if isinstance(acronyms, str): acronyms = [acronyms]
    
# 1.2. ---------------------------------------------------------------------------- Map the primary zone name to itself
    acronym_to_zone[zone] = zone

# 1.3. ------------------------------------------------------------------ Map all sub-acronyms to the primary zone name
    for acr in acronyms:
        acronym_to_zone[acr] = zone

# 2. ---------------------------- Sort by length (descending) to ensure longer acronyms are matched before shorter ones
all_patterns = sorted(acronym_to_zone.keys(), key=len, reverse=True)
pattern = "|".join(map(re.escape, all_patterns))

def process_header(header):
    
# 2.1. -------------------------------------------------------- Find all matches in the order they appear in the string
    matches = re.findall(pattern, header)
    
    if not matches:
        
# 2.2. -------------------------------------------------------- If no zone names or acronyms found, return header as is
        return header
    
# 2.3. ----------------- Map found acronyms to their primary zone names; Use a list to maintain the order of appearance
    cleaned_zones = []
    for m in matches:
        zone = acronym_to_zone[m]
        cleaned_zones.append(zone)
    
# 2.4. ----------------------------------------------------------------------------------------------- Join with ' -> '
    return " -> ".join(cleaned_zones)

# 3. Apply the transformation to the column names
cross_border_flows_raw_data_df.columns = [
    process_header(col) for col in cross_border_flows_raw_data_df.columns
]

# 4. ------------------------------------------------------------------------------------------------------------ Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,BE -> NL,DE -> FR,DE -> NL,DE -> BE,UK -> NL,UK -> FR,UK -> UK,UK -> UK,UK -> FR,UK -> FR,FR -> UK,UK -> FR,UK -> DE,UK -> NL,UK -> BE
0,2013-01-01 00:00:00,-104.107714,-3920.604294,-72.693467,-4735.092357,0.003487,2460.370002,4920.655273,0.003112,0.003286,3444.273289,3645.001102,0.001359,3444.333691,3444.519089,2952.444930,2460.370959
1,2013-01-01 01:00:00,-248.143043,-3914.429679,-173.266592,-4722.658899,0.003448,2460.370012,4920.655296,0.003206,0.003379,3444.273312,3645.001124,0.001353,3444.333714,3444.519108,2952.444935,2460.370984
2,2013-01-01 02:00:00,-515.167759,-3899.539575,-359.717472,-4705.476232,0.003433,2460.369989,4920.655292,0.003215,0.003374,3444.273310,3645.001122,0.001353,3444.333712,3444.519106,2952.444914,2460.370980
3,2013-01-01 03:00:00,-849.520353,-3891.289627,-593.180284,-4703.412134,0.003380,2460.369936,4920.655283,0.003193,0.003336,3444.273301,3645.001113,0.001358,3444.333703,3444.519105,2952.444859,2460.370965
4,2013-01-01 04:00:00,-649.203756,-3818.788794,-453.308563,-4615.770263,0.003628,2460.370606,4920.655295,0.003223,0.003362,3444.273317,3645.001128,0.001345,3444.333719,3444.519095,2952.445527,2460.371025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1141.823671,-7630.706649,797.282221,-9114.841750,0.002658,2460.370501,4920.655535,0.004600,0.004648,3444.273570,3645.001379,0.001169,3444.333972,3444.519324,2952.445420,2460.371264
8756,2013-12-31 20:00:00,1715.916067,-7723.255226,1198.144114,-9232.381190,0.002634,2460.370485,4920.655522,0.004584,0.004599,3444.273559,3645.001369,0.001174,3444.333961,3444.519321,2952.445404,2460.371252
8757,2013-12-31 21:00:00,2387.612570,-8251.781594,1667.158398,-9889.501646,0.002603,2460.370466,4920.655511,0.004582,0.004584,3444.273550,3645.001359,0.001179,3444.333952,3444.519320,2952.445384,2460.371241
8758,2013-12-31 22:00:00,2457.405473,-7843.638983,1715.891528,-9415.943669,0.002698,2460.370656,4920.655520,0.003625,0.003660,3444.273560,3645.001369,0.001170,3444.333962,3444.519316,2952.445573,2460.371275


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Aggregating repeated columns.
    </div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [18]:
# 1. --------------------------------------------------------- Assuming cross_border_flows_raw_data_df is the DataFrame
df = cross_border_flows_raw_data_df.copy()

# 2. -------------------------------------------------------------------------------------- Find duplicate column names
duplicate_columns = df.columns[df.columns.duplicated(keep=False)].unique()

# 3. ------------------------------------------- For each duplicate column name, sum the values and create a new column
for col in duplicate_columns:

# 3.1. -------------------------------------------------------------------------- Get all columns with the current name
    cols_to_sum = df.loc[:, col].columns
    
# 3.2. ------------------------------------------------------------------------- Sum the values and create a new column
    df[col + '_sum'] = df[cols_to_sum].sum(axis=1)
    
# 3.3. -------------------------------------------------------------------------------------- Drop the original columns
    df = df.drop(columns=cols_to_sum)

# 4. ------------------------------------------------------------------- Rename the summed columns to the original name
df = df.rename(columns={col + '_sum': col for col in duplicate_columns})

# 5. ------------------------------------------------------------------------------------------- Result is stored in df
cross_border_flows_raw_data_df = df

# 6. ------------------------------------------------------------------------------------------------------------- Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,BE -> NL,DE -> FR,DE -> NL,DE -> BE,FR -> UK,UK -> DE,UK -> BE,UK -> NL,UK -> FR,UK -> UK
0,2013-01-01 00:00:00,-104.107714,-3920.604294,-72.693467,-4735.092357,0.003487,0.001359,3444.519089,2460.370959,10825.629865,61817.053416,0.012796
1,2013-01-01 01:00:00,-248.143043,-3914.429679,-173.266592,-4722.658899,0.003448,0.001353,3444.519108,2460.370984,10825.629894,61817.053784,0.013170
2,2013-01-01 02:00:00,-515.167759,-3899.539575,-359.717472,-4705.476232,0.003433,0.001353,3444.519106,2460.370980,10825.629807,61817.053750,0.013179
3,2013-01-01 03:00:00,-849.520353,-3891.289627,-593.180284,-4703.412134,0.003380,0.001358,3444.519105,2460.370965,10825.629589,61817.053604,0.013059
4,2013-01-01 04:00:00,-649.203756,-3818.788794,-453.308563,-4615.770263,0.003628,0.001345,3444.519095,2460.371025,10825.632265,61817.053833,0.013169
...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1141.823671,-7630.706649,797.282221,-9114.841750,0.002658,0.001169,3444.519324,2460.371264,10825.631842,61817.057822,0.018496
8756,2013-12-31 20:00:00,1715.916067,-7723.255226,1198.144114,-9232.381190,0.002634,0.001174,3444.519321,2460.371252,10825.631779,61817.057645,0.018366
8757,2013-12-31 21:00:00,2387.612570,-8251.781594,1667.158398,-9889.501646,0.002603,0.001179,3444.519320,2460.371241,10825.631700,61817.057488,0.018333
8758,2013-12-31 22:00:00,2457.405473,-7843.638983,1715.891528,-9415.943669,0.002698,0.001170,3444.519316,2460.371275,10825.632457,61817.057643,0.014570


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Separating import and export flows.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [19]:
# 1. ---------------------------------------------------------------------- Make a copy to avoid modifying the original
df = cross_border_flows_raw_data_df.copy()

# 2. ----------------------------------------------------------------------- Identify flow columns vs. non-flow columns
flow_columns = []
non_flow_columns = []

for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            source = parts[0].strip()
            target = parts[1].strip()
            
# 2.1. ------------------------------------------- Only consider it a flow column if both zones are valid and different
            if source in zone_names and target in zone_names and source != target:
                flow_columns.append(col)
                continue
                
# 2.2. ------------------------------------------------------------------ If not a valid flow column, treat as non-flow
    non_flow_columns.append(col)

# 3. ---------------------------------------------------------------------- build new derived columns from flow_columns
new_columns = {}  # maps column name -> list of Series (to allow duplicates)

# 4. ----------------------------------------------------------------------------------------- Process each flow column
for col in flow_columns:
    source, target = map(str.strip, col.split(' -> '))
    
# 4.1. ---------------------------------------------------------------------------------- Forward: positive values only
    forward_name = f"{source} -> {target}"
    forward_values = df[col].where(df[col] > 0)
    
# 4.2. ---------------------------------------------------------------------- Reverse: absolute of negative values only
    reverse_name = f"{target} -> {source}"
    reverse_values = (-df[col]).where(df[col] < 0)
    
# 4.3 --------------------------------------------------------- Append to lists (allowing duplicate column names later)
    new_columns[forward_name] = new_columns.get(forward_name, []) + [forward_values]
    new_columns[reverse_name] = new_columns.get(reverse_name, []) + [reverse_values]

# 5. ----------------------------------- Build final column list: first non-flow columns, then all derived flow columns
series_list = []
col_names = []

# 6. --------------------------------------------------------------------------------------- Add non-flow columns as-is
for col in non_flow_columns:
    series_list.append(df[col].reset_index(drop=True))
    col_names.append(col)

# 7. -------------------------------------------------------------- Add all derived flow columns (including duplicates)
for col_name, series_group in new_columns.items():
    for s in series_group:
        series_list.append(s.reset_index(drop=True))
        col_names.append(col_name)

# 8. ------------------------------------------------ Construct final DataFrame with intentional duplicate column names
final_array = np.column_stack([s.values for s in series_list])
cross_border_flows_raw_data_df = pd.DataFrame(final_array, columns=col_names, index=df.index)

# 9. ------------------------------------------------------------------------------------------------------------- Done 
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,UK -> UK,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,...,FR -> UK,FR -> UK,UK -> FR,UK -> FR,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK
0,2013-01-01 00:00:00,0.012796,NaN,104.107714,NaN,3920.604294,NaN,72.693467,NaN,4735.092357,...,0.001359,NaN,NaN,61817.053416,3444.519089,NaN,2460.370959,NaN,10825.629865,NaN
1,2013-01-01 01:00:00,0.01317,NaN,248.143043,NaN,3914.429679,NaN,173.266592,NaN,4722.658899,...,0.001353,NaN,NaN,61817.053784,3444.519108,NaN,2460.370984,NaN,10825.629894,NaN
2,2013-01-01 02:00:00,0.013179,NaN,515.167759,NaN,3899.539575,NaN,359.717472,NaN,4705.476232,...,0.001353,NaN,NaN,61817.05375,3444.519106,NaN,2460.37098,NaN,10825.629807,NaN
3,2013-01-01 03:00:00,0.013059,NaN,849.520353,NaN,3891.289627,NaN,593.180284,NaN,4703.412134,...,0.001358,NaN,NaN,61817.053604,3444.519105,NaN,2460.370965,NaN,10825.629589,NaN
4,2013-01-01 04:00:00,0.013169,NaN,649.203756,NaN,3818.788794,NaN,453.308563,NaN,4615.770263,...,0.001345,NaN,NaN,61817.053833,3444.519095,NaN,2460.371025,NaN,10825.632265,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,0.018496,1141.823671,NaN,NaN,7630.706649,797.282221,NaN,NaN,9114.84175,...,0.001169,NaN,NaN,61817.057822,3444.519324,NaN,2460.371264,NaN,10825.631842,NaN
8756,2013-12-31 20:00:00,0.018366,1715.916067,NaN,NaN,7723.255226,1198.144114,NaN,NaN,9232.38119,...,0.001174,NaN,NaN,61817.057645,3444.519321,NaN,2460.371252,NaN,10825.631779,NaN
8757,2013-12-31 21:00:00,0.018333,2387.61257,NaN,NaN,8251.781594,1667.158398,NaN,NaN,9889.501646,...,0.001179,NaN,NaN,61817.057488,3444.51932,NaN,2460.371241,NaN,10825.6317,NaN
8758,2013-12-31 22:00:00,0.01457,2457.405473,NaN,NaN,7843.638983,1715.891528,NaN,NaN,9415.943669,...,0.00117,NaN,NaN,61817.057643,3444.519316,NaN,2460.371275,NaN,10825.632457,NaN


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Separating import and export flows.
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [20]:
# 1. ------------------------------------------------------------------------------------- Make a copy of the DataFrame
df = cross_border_flows_raw_data_df.copy()

# 2. ----------------------------------------------------------------------------- REMOVE SELF-LOOP COLUMNS: "AA -> AA"
columns_to_drop = []
for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            source, target = parts[0].strip(), parts[1].strip()
            if source == target and source in zone_names:
                columns_to_drop.append(col)

# 3. --------------------------------------------------------------------------------------- Drop all self-loop columns
df = df.drop(columns=columns_to_drop)

# 4. ------------------------------------------------------------------ Get all column names that appear more than once
duplicated_names = df.columns[df.columns.duplicated(keep=False)].unique()

# 5. ------------------------------------------------------------------ For each duplicated name, sum all its instances
for name in duplicated_names:
    
# 5.1. ------------------------------------------------------------------------------ Select all columns with this name
    subset = df.filter(regex=f'^{name}$')  # Exact match (avoids partial matches)
    # Sum across columns
    df[name + '_sum'] = subset.sum(axis=1)
    # Drop all original instances
    df = df.drop(columns=subset.columns)

# 6. ---------------------------------------------------------------------- RENAME summed columns back to original name
rename_map = {name + '_sum': name for name in duplicated_names}
df = df.rename(columns=rename_map)

# 7. ------------------------------------------------------------------------------------ Update the original DataFrame
cross_border_flows_raw_data_df = df

# 8. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.")
cross_border_flows_raw_data_df

✅ Processing complete. Updated DataFrame stored in 'cross_border_flows_raw_data_df'.


,snapshot,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,DE -> BE,BE -> DE,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK,FR -> UK,UK -> FR
0,2013-01-01 00:00:00,NaN,104.107714,NaN,3920.604294,NaN,72.693467,NaN,4735.092357,0.003487,NaN,3444.519089,NaN,2460.370959,NaN,10825.629865,NaN,0.001359,61817.053416
1,2013-01-01 01:00:00,NaN,248.143043,NaN,3914.429679,NaN,173.266592,NaN,4722.658899,0.003448,NaN,3444.519108,NaN,2460.370984,NaN,10825.629894,NaN,0.001353,61817.053784
2,2013-01-01 02:00:00,NaN,515.167759,NaN,3899.539575,NaN,359.717472,NaN,4705.476232,0.003433,NaN,3444.519106,NaN,2460.37098,NaN,10825.629807,NaN,0.001353,61817.05375
3,2013-01-01 03:00:00,NaN,849.520353,NaN,3891.289627,NaN,593.180284,NaN,4703.412134,0.00338,NaN,3444.519105,NaN,2460.370965,NaN,10825.629589,NaN,0.001358,61817.053604
4,2013-01-01 04:00:00,NaN,649.203756,NaN,3818.788794,NaN,453.308563,NaN,4615.770263,0.003628,NaN,3444.519095,NaN,2460.371025,NaN,10825.632265,NaN,0.001345,61817.053833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1141.823671,NaN,NaN,7630.706649,797.282221,NaN,NaN,9114.84175,0.002658,NaN,3444.519324,NaN,2460.371264,NaN,10825.631842,NaN,0.001169,61817.057822
8756,2013-12-31 20:00:00,1715.916067,NaN,NaN,7723.255226,1198.144114,NaN,NaN,9232.38119,0.002634,NaN,3444.519321,NaN,2460.371252,NaN,10825.631779,NaN,0.001174,61817.057645
8757,2013-12-31 21:00:00,2387.61257,NaN,NaN,8251.781594,1667.158398,NaN,NaN,9889.501646,0.002603,NaN,3444.51932,NaN,2460.371241,NaN,10825.6317,NaN,0.001179,61817.057488
8758,2013-12-31 22:00:00,2457.405473,NaN,NaN,7843.638983,1715.891528,NaN,NaN,9415.943669,0.002698,NaN,3444.519316,NaN,2460.371275,NaN,10825.632457,NaN,0.00117,61817.057643


<div style="background-color: black;">
<hr style="border: 0.5px solid skyblue;">
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Cross-border flow direction check
</div>
<hr style="border: 0.5px solid skyblue;">
</div>

In [21]:
# 1. ------------------------------------------------------------------------------------------------- Get the raw data
df = cross_border_flows_raw_data_df.copy()

# 2. ------------------------------------------------------------------------------- Identify flow and non-flow columns
flow_cols = []
non_flow_cols = []

for col in df.columns:
    if ' -> ' in col:
        parts = col.split(' -> ')
        if len(parts) == 2:
            src = parts[0].strip()
            tgt = parts[1].strip()
            if src in zone_names and tgt in zone_names and src != tgt:
                flow_cols.append(col)
                continue
    non_flow_cols.append(col)

# 3. ---------------------------------------------------------------------------------- Ensure flow columns are numeric
for col in flow_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # Converts to float64, NaN for non-numeric

flow_col_set = set(flow_cols)
processed = set()

for col in flow_cols:
    src, tgt = map(str.strip, col.split(' -> '))
    mirror = f"{tgt} -> {src}"
    pair_key = tuple(sorted([src, tgt]))
    
    if mirror in flow_col_set and pair_key not in processed:
        processed.add(pair_key)
        
# 3.1. -------------------------------------------------------------- Now safe to fill NaN with 0 (columns are numeric)
        forward = df[col].fillna(0)
        reverse = df[mirror].fillna(0)
        
        net = forward - reverse
        df[col] = np.where(net >= 0, net, 0)
        df[mirror] = np.where(net < 0, -net, 0)

# 4. ------------------------------------------------------------------------------------------------------------- Done
cross_border_flows_raw_data_df = df
print("✅ Net directional flows computed successfully.")
cross_border_flows_raw_data_df

✅ Net directional flows computed successfully.


,snapshot,BE -> FR,FR -> BE,BE -> NL,NL -> BE,DE -> FR,FR -> DE,DE -> NL,NL -> DE,DE -> BE,BE -> DE,UK -> DE,DE -> UK,UK -> BE,BE -> UK,UK -> NL,NL -> UK,FR -> UK,UK -> FR
0,2013-01-01 00:00:00,0.000000,104.107714,0.0,3920.604294,0.000000,72.693467,0.0,4735.092357,0.003487,0.0,3444.519089,0.0,2460.370959,0.0,10825.629865,0.0,0.0,61817.052057
1,2013-01-01 01:00:00,0.000000,248.143043,0.0,3914.429679,0.000000,173.266592,0.0,4722.658899,0.003448,0.0,3444.519108,0.0,2460.370984,0.0,10825.629894,0.0,0.0,61817.052431
2,2013-01-01 02:00:00,0.000000,515.167759,0.0,3899.539575,0.000000,359.717472,0.0,4705.476232,0.003433,0.0,3444.519106,0.0,2460.370980,0.0,10825.629807,0.0,0.0,61817.052397
3,2013-01-01 03:00:00,0.000000,849.520353,0.0,3891.289627,0.000000,593.180284,0.0,4703.412134,0.003380,0.0,3444.519105,0.0,2460.370965,0.0,10825.629589,0.0,0.0,61817.052246
4,2013-01-01 04:00:00,0.000000,649.203756,0.0,3818.788794,0.000000,453.308563,0.0,4615.770263,0.003628,0.0,3444.519095,0.0,2460.371025,0.0,10825.632265,0.0,0.0,61817.052488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2013-12-31 19:00:00,1141.823671,0.000000,0.0,7630.706649,797.282221,0.000000,0.0,9114.841750,0.002658,0.0,3444.519324,0.0,2460.371264,0.0,10825.631842,0.0,0.0,61817.056653
8756,2013-12-31 20:00:00,1715.916067,0.000000,0.0,7723.255226,1198.144114,0.000000,0.0,9232.381190,0.002634,0.0,3444.519321,0.0,2460.371252,0.0,10825.631779,0.0,0.0,61817.056471
8757,2013-12-31 21:00:00,2387.612570,0.000000,0.0,8251.781594,1667.158398,0.000000,0.0,9889.501646,0.002603,0.0,3444.519320,0.0,2460.371241,0.0,10825.631700,0.0,0.0,61817.056309
8758,2013-12-31 22:00:00,2457.405473,0.000000,0.0,7843.638983,1715.891528,0.000000,0.0,9415.943669,0.002698,0.0,3444.519316,0.0,2460.371275,0.0,10825.632457,0.0,0.0,61817.056473


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 17px; font-family: TimesNewRoman; color:skyblue">
        6.3. Total Cross Border Flows Values
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Getting total Corss Boder Flows for all country pairs.
<hr style="border: 0.5px solid skyblue;">
</div>

In [22]:
# 1. ---------------------------------------------------------------------------------------- Identify time-step column
def find_time_column(df):
    candidates = []

    for col in df.columns:

# 1.1. -------------------------------------------------------------------------------------------- Must NOT be numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            continue

# 1.2. --------------------------------------------------------------------------------- Try datetime conversion safely
        converted = pd.to_datetime(df[col], errors="coerce")

# 1.3. --------------------------------------------------------------------------------------------- Must fully convert
        if not converted.notna().all():
            continue

# 1.4. ------------------------------------------------------------------------------- Must have many unique timestamps
        if converted.nunique() < 100:
            continue

# 1.5. --------------------------------------------------------------------------------------- Must vary in hour or day
        if converted.dt.hour.nunique() <= 1 and converted.dt.day.nunique() <= 1:
            continue

        candidates.append(col)

    if len(candidates) != 1:
        raise ValueError(
            f"Expected exactly one time column, found {len(candidates)}: {candidates}"
        )

    return candidates[0]

# 2. ------------------------------------------------------------------------------- Normalize dataframe to target year
def normalize_year(df, target_year):
    df = df.copy()

    time_col = find_time_column(df)

# 2.1. ------------------------------------------------------------------------- FORCE timezone consistency (remove tz)
    df[time_col] = pd.to_datetime(df[time_col], utc=True).dt.tz_localize(None)

    df = df.set_index(time_col).sort_index()

# 2.2. -------------------------------------------------------------------------------- Shift timestamps to target year
    df.index = df.index.map(lambda ts: ts.replace(year=target_year))

# 2.3. ----------------------------------------------------------------------------------- Remove duplicated timestamps
    df = df[~df.index.duplicated(keep="first")]

# 2.4. ------------------------------------------------------------------- Full hourly index for target year (tz-naive)
    start = pd.Timestamp(f"{target_year}-01-01 00:00:00")
    end = pd.Timestamp(f"{target_year}-12-31 23:00:00")
    full_index = pd.date_range(start=start, end=end, freq="h")

    df_full = pd.DataFrame(index=full_index, columns=df.columns)

# 2.5. ------------------------------------------------------------------------------------------- Hour-preserving fill
    for hour in range(24):
        src = df[df.index.hour == hour]
        tgt_index = df_full.index[df_full.index.hour == hour]

        if not src.empty:
            filled = src.reindex(tgt_index, method="nearest")
            df_full.loc[tgt_index] = filled.values

    df_full = df_full.reset_index().rename(columns={"index": time_col})

    return df_full, time_col

# 3. ---------------------------------------------------------------------------------------- Normalize both dataframes
cross_border_flows_df, time_col_1 = normalize_year(
    cross_border_flows_df, data_target_year
)

cross_border_flows_raw_data_df, time_col_2 = normalize_year(
    cross_border_flows_raw_data_df, data_target_year
)

# 4. ------------------------------------------------------------------------------------------------ Verify row counts
if len(cross_border_flows_df) != len(cross_border_flows_raw_data_df):
    raise ValueError("Row count mismatch after normalization.")

# 5. ---------------------------------------------------------------------------- Identify bigger and smaller dataframe
if cross_border_flows_df.shape[1] >= cross_border_flows_raw_data_df.shape[1]:
    df_big = cross_border_flows_df
    df_small = cross_border_flows_raw_data_df
    time_col = time_col_1
else:
    df_big = cross_border_flows_raw_data_df
    df_small = cross_border_flows_df
    time_col = time_col_2

# 6. -------------------------------------------------------------------- Copy matching columns (excluding time column)
common_columns = (
    set(df_small.columns)
    .intersection(df_big.columns)
    .difference({time_col})
)

for col in common_columns:
    df_big[col] = df_small[col].values

# 7. ------------------------------------------------------------------- Replace zeros with NaN (excluding time column)
flow_columns = df_big.columns.difference([time_col])

df_big.loc[:, flow_columns] = (
    df_big.loc[:, flow_columns]
    .replace(0, np.nan)
    .infer_objects(copy=False)
)

# 8. ------------------------------------------------------------------------------------------------------------- Done
print("✅ Net directional flows computed successfully.")
cross_border_flows_df

✅ Net directional flows computed successfully.


/tmp/ipykernel_2404533/3187185117.py:110: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(0, np.nan)


,index,FR -> RoW,FR -> NL,DE -> FR,DE -> RoW,BE -> RoW,FR -> BE,FR -> UK,NL -> UK,UK -> RoW,...,UK -> NL,UK -> BE,RoW -> DE,RoW -> NL,NL -> RoW,NL -> DE,DE -> UK,RoW -> FR,UK -> DE,FR -> DE
0,2050-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,104.107714,NaN,NaN,NaN,...,10825.629865,2460.370959,NaN,NaN,NaN,4735.092357,NaN,NaN,3444.519089,72.693467
1,2050-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,248.143043,NaN,NaN,NaN,...,10825.629894,2460.370984,NaN,NaN,NaN,4722.658899,NaN,NaN,3444.519108,173.266592
2,2050-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,515.167759,NaN,NaN,NaN,...,10825.629807,2460.37098,NaN,NaN,NaN,4705.476232,NaN,NaN,3444.519106,359.717472
3,2050-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,849.520353,NaN,NaN,NaN,...,10825.629589,2460.370965,NaN,NaN,NaN,4703.412134,NaN,NaN,3444.519105,593.180284
4,2050-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,649.203756,NaN,NaN,NaN,...,10825.632265,2460.371025,NaN,NaN,NaN,4615.770263,NaN,NaN,3444.519095,453.308563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2050-12-31 19:00:00,NaN,NaN,797.282221,NaN,NaN,NaN,NaN,NaN,NaN,...,10825.631842,2460.371264,NaN,NaN,NaN,9114.84175,NaN,NaN,3444.519324,NaN
8756,2050-12-31 20:00:00,NaN,NaN,1198.144114,NaN,NaN,NaN,NaN,NaN,NaN,...,10825.631779,2460.371252,NaN,NaN,NaN,9232.38119,NaN,NaN,3444.519321,NaN
8757,2050-12-31 21:00:00,NaN,NaN,1667.158398,NaN,NaN,NaN,NaN,NaN,NaN,...,10825.6317,2460.371241,NaN,NaN,NaN,9889.501646,NaN,NaN,3444.51932,NaN
8758,2050-12-31 22:00:00,NaN,NaN,1715.891528,NaN,NaN,NaN,NaN,NaN,NaN,...,10825.632457,2460.371275,NaN,NaN,NaN,9415.943669,NaN,NaN,3444.519316,NaN


<div style="background-color: black;">
    <hr style="border: 2px solid skyblue;">
    <div style="text-align: justify; margin-left: 3.0em; font-weight: bold; font-size: 16px; font-family: TimesNewRoman; color:skyblue">
6.4. Formated NTC Data Files
    </div>
    <div style="text-align: justify; margin-left: 0.0em; font-weight: unbold; font-size: 14px; font-family: TimesNewRoman; color:skyblue">
Storing the formatted and disaggregated NTC data into the designated directory within the Dispa-SET folder structure.
    <hr style="border: 0.5px solid skyblue;">
</div>

In [23]:
# 1. -------------------------------------------------------------------------------------- Ensure the directory exists
os.makedirs(cross_border_flows_pypsa_formated_data_folder_path, exist_ok=True)

# 2. ----------------------------------------------------------------------------------------- Define the CSV file path
csv_filename = f"{data_target_year}.csv"
csv_filepath = os.path.join(cross_border_flows_pypsa_formated_data_folder_path, csv_filename)

# 3. ---------------------------------------------------------------------------------------- Save the DataFrame to CSV
cross_border_flows_df.to_csv(csv_filepath, index=False)

# 4. ------------------------------------------------------------------------------------------------------------- Done
print(f"DataFrame successfully saved to: {csv_filepath}")

DataFrame successfully saved to: /home/ray/Dispa-SET_Unleash/Database_PyPSA/Suficiency_Scenario/CrossBorderFlows/1h/2050.csv


<div style="background-color: black;">
<hr style="border: 4px solid skyblue;">
</div>